In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from global_vars import chargement_df

In [ ]:
df = pd.read_csv("../data/raw/accidents_2019_2023.csv")

In [ ]:
# changement ordre de gravité pour avoir un ordre logique 
df['grav'] = df['grav'].replace({2: 42})
df['grav'] = df['grav'].replace({4: 2})
df['grav'] = df['grav'].replace({42: 4})

In [ ]:
# création d'une variable age au moment de l'accident
df['age'] = df['an']-df['an_nais']
df.drop(['an_nais'],axis=1,inplace = True)

In [ ]:
# suppression des valeurs manquantes pour la variable cible

df = df[accidents_copy['grav'] != -1]

In [ ]:
# Définition de la fonction pour homogénéiser le format de l'heure
def homogenize_hour_format(row):
    # Convertir l'heure en chaîne de caractères
    hour_str = str(row['hrmn'])
    
    # Si l'année est comprise entre 2005 et 2018, ajuster le format de l'heure
    if row['an'] < 2019:
        # Extraire les deux derniers chiffres pour les minutes
        minutes = hour_str[-2:].zfill(2)
        
        # extrait les autre pour les heures
        hour = hour_str[:-2].zfill(2)
        
        #concatene avec ':' pour obtenir le format 'HH:MM'
        return f'{hour}:{minutes}'
    
    # Si l'année est 2019 ou plus, le format est déjà 'HH:MM'
    return hour_str

# Appliquer la fonction à la colonne 'hrmn'
df['hrmn'] = df.apply(homogenize_hour_format, axis=1)

In [ ]:
# remplacement des valeurs manquantes de lum en fonction des heures de la journée

# Remplacer les -1 par 1 entre 9h et 17h
df.loc[(df['lum'] == -1) &
                   (df['hrmn'] > '09:00') &
                   (df['hrmn'] < '17:00'), 'lum'] = 1

# Remplacer les -1 par 3 avant 7h ou après 20h
df.loc[(df['lum'] == -1) &
                   ((df['hrmn'] < '07:00') |
                    (df['hrmn'] > '20:00')), 'lum'] = 3

# Remplacer les -1 restants par 2
df['lum'] = df['lum'].replace({-1: 2})

In [ ]:
# Création d'une colonne 'date' à partir de jour/mois/an
df['date'] = pd.to_datetime(dict(year=df['an'], month=df['mois'], day=df['jour']), errors='coerce')

# Création du jour de la semaine
df['jour_semaine'] = df['date'].dt.dayofweek  

# Extraction de l'heure depuis la colonne 'hrmn' s
def extraire_heure(chaine):
    try:
        return int(str(chaine).split(':')[0])
    except:
        return np.nan

df['heure'] = df['hrmn'].apply(extraire_heure)

In [ ]:
# Encodage des variables catégorielles
cat_cols = df_clean.select_dtypes(include='object').columns


encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_clean[cat_cols] = encoder.fit_transform(df_clean[cat_cols])

In [ ]:
# Création de nouvelles variables 
# 1. Heure de l'accident 
def periode_jour(heure):
    if pd.isna(heure): return np.nan
    h = int(str(heure).split(':')[0])
    if 5 <= h < 12:
        return "matin"
    elif 12 <= h < 18:
        return "après-midi"
    elif 18 <= h < 22:
        return "soir"
    else:
        return "nuit"

df_clean['periode_jour'] = df_clean['hrmn'].apply(periode_jour)

# 2. Num_Acc → nombre de véhicules par accident
nb_vehicules = df_clean.groupby("Num_Acc")["id_vehicule"].nunique().reset_index()
nb_vehicules.columns = ["Num_Acc", "nb_vehicules"]
df_clean = df_clean.merge(nb_vehicules, on="Num_Acc", how="left")